In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vit_timesformer_probe import TimeSformer, TimeSformerLinearProbe
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="fullshot"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:1' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-07 22:06:13,514]::3493104939::DEBUG::=============fullshot+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:1


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
mtcnn= MTCNN(image_size=224, margin=0, min_face_size= 20, thresholds=[0.6, 0.7, 0.7], post_process=True)
print(1)

1


In [5]:
validation_set_data=getdata("/data/qx/datafrom136/data/15Frames/Full_frames/valid_set.dat","valid")
validation_set_data=reshape_to_expected_input(validation_set_data)
print(1)

1


In [6]:
train_set_data=getdata("/data/qx/datafrom136/data/15Frames/Full_frames/train_set.dat","train")
train_set_data=reshape_to_expected_input(train_set_data)
print(1)

1


In [7]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [8]:
class ChalearnDataset(Dataset):
    def __init__(self,fullshot,tagdata,transform=None):
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return fullshot,big_five_sorces
print(1)

1


In [9]:
num_workerssz = 20
lr = 3e-05
epochs = 120

# Multi-seed setting
SEEDS = [42, 123, 777, 1004, 2024]
patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print(1)


1


In [10]:
train_set_data = ChalearnDataset(
    fullshot=train_set_data[0],
    tagdata=train_set_data[1],
    transform=transform
)

val_set_data = ChalearnDataset(
    fullshot=validation_set_data[0],
    tagdata=validation_set_data[1],
    transform=transform
)

print(1)


1


In [11]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [12]:
# ============================================================
# TimeSformer Video-only Block-wise Linear Probe
# ============================================================
#
# Probe targets:
#   block1 ... block12 : CLS token after each complete TimeSformer block
#   final_norm         : CLS token after final LayerNorm
#
# Backbone is completely frozen.
# Only independent Linear(768, 5) probe heads are optimized.
# ============================================================

PROBE_SEED = 42
PROBE_LR = 1e-3
WEIGHT_DECAY = 1e-4
PROBE_EPOCHS = 120
PATIENCE = 20

BACKBONE_CHECKPOINT = "./save_timesformer_folder/model_best_multiseed.pth"


def set_probe_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def probe_seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_probe_seed(PROBE_SEED)

generator = torch.Generator()
generator.manual_seed(PROBE_SEED)

train_dataloader = DataLoader(
    dataset=train_set_data,
    batch_size=batchsz,
    shuffle=True,
    num_workers=num_workerssz,
    worker_init_fn=probe_seed_worker,
    generator=generator
)

val_dataloader = DataLoader(
    dataset=val_set_data,
    batch_size=batchsz,
    shuffle=False,
    num_workers=num_workerssz,
    worker_init_fn=probe_seed_worker,
    generator=generator
)

# ------------------------------------------------------------
# 1. Re-create EXACTLY the same TimeSformer backbone
#    used by the supplied original notebook.
# ------------------------------------------------------------
backbone = TimeSformer().to(device)

checkpoint = torch.load(
    BACKBONE_CHECKPOINT,
    map_location=device
)

state_dict = (
    checkpoint["model"]
    if isinstance(checkpoint, dict) and "model" in checkpoint
    else checkpoint
)

backbone.load_state_dict(
    state_dict,
    strict=True
)

# ------------------------------------------------------------
# 2. Frozen backbone + independent linear heads
# ------------------------------------------------------------
model = TimeSformerLinearProbe(
    backbone=backbone,
    dim=768,
    num_outputs=5
).to(device)

backbone_trainable = sum(
    p.numel()
    for p in model.backbone.parameters()
    if p.requires_grad
)

probe_trainable = sum(
    p.numel()
    for p in model.probe_heads.parameters()
    if p.requires_grad
)

print(f"Backbone trainable parameters: {backbone_trainable}")
print(f"Probe trainable parameters:    {probe_trainable}")

assert backbone_trainable == 0, "Backbone must remain frozen."

criterion = nn.L1Loss().to(device)

optimizer = torch.optim.AdamW(
    model.probe_heads.parameters(),
    lr=PROBE_LR,
    weight_decay=WEIGHT_DECAY
)

probe_names = list(model.probe_heads.keys())

best_mae = {
    name: float("inf")
    for name in probe_names
}

best_epoch = {
    name: -1
    for name in probe_names
}

best_mean_val_mae = float("inf")
no_improve_count = 0

for epoch in range(PROBE_EPOCHS):

    # ========================================================
    # Train probe heads only
    # ========================================================
    model.train()

    train_abs_sum = {
        name: 0.0
        for name in probe_names
    }
    train_count = 0

    for fullshot, big_five_data in tqdm(
        train_dataloader,
        desc=f"Probe Train {epoch + 1}/{PROBE_EPOCHS}"
    ):
        fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)

        # Preserve the original label layout while avoiding accidental
        # batch-dimension removal when batch size == 1.
        big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)

        optimizer.zero_grad()

        outputs = model(fullshot)

        losses = {
            name: criterion(outputs[name], big_five_data)
            for name in probe_names
        }

        total_loss = sum(losses.values())
        total_loss.backward()
        optimizer.step()

        scalar_count = big_five_data.numel()
        train_count += scalar_count

        for name in probe_names:
            train_abs_sum[name] += torch.abs(
                outputs[name].detach() - big_five_data
            ).sum().item()

    train_mae = {
        name: train_abs_sum[name] / train_count
        for name in probe_names
    }

    # ========================================================
    # Validation
    # ========================================================
    model.eval()

    val_abs_sum = {
        name: 0.0
        for name in probe_names
    }
    val_count = 0

    with torch.no_grad():

        for fullshot, big_five_data in tqdm(
            val_dataloader,
            desc=f"Probe Valid {epoch + 1}/{PROBE_EPOCHS}"
        ):
            fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
            big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)

            outputs = model(fullshot)

            val_count += big_five_data.numel()

            for name in probe_names:
                val_abs_sum[name] += torch.abs(
                    outputs[name] - big_five_data
                ).sum().item()

    val_mae = {
        name: val_abs_sum[name] / val_count
        for name in probe_names
    }

    # Each probe independently tracks its own best epoch.
    for name in probe_names:
        if val_mae[name] < best_mae[name]:
            best_mae[name] = val_mae[name]
            best_epoch[name] = epoch + 1

    # Global early stopping criterion = mean validation MAE
    # over all independent probe heads.
    mean_val_mae = float(
        np.mean([val_mae[name] for name in probe_names])
    )

    if mean_val_mae < best_mean_val_mae:
        best_mean_val_mae = mean_val_mae
        no_improve_count = 0
    else:
        no_improve_count += 1

    print("\n" + "-" * 90)
    print(
        f"Epoch {epoch + 1:03d} | "
        f"Mean Val MAE={mean_val_mae:.6f} | "
        f"Patience={no_improve_count}/{PATIENCE}"
    )

    for name in probe_names:
        print(
            f"{name:<12} | "
            f"train MAE {train_mae[name]:.6f} | "
            f"val MAE {val_mae[name]:.6f} | "
            f"1-MAE {1.0 - val_mae[name]:.6f} | "
            f"best 1-MAE {1.0 - best_mae[name]:.6f}"
        )

    if no_improve_count >= PATIENCE:
        print(
            f"\nEarly stopping at epoch {epoch + 1}. "
            f"Best mean validation MAE={best_mean_val_mae:.6f}"
        )
        break

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# Final Probe Summary
# ============================================================

print("\n" + "=" * 90)
print("FINAL TIMESFORMER VIDEO-ONLY LINEAR PROBE RESULTS")
print("=" * 90)
print(
    f"{'Representation':<18}"
    f"{'Best Epoch':>12}"
    f"{'MAE':>14}"
    f"{'1-MAE':>14}"
)
print("-" * 90)

for name in probe_names:
    print(
        f"{name:<18}"
        f"{best_epoch[name]:>12d}"
        f"{best_mae[name]:>14.6f}"
        f"{1.0 - best_mae[name]:>14.6f}"
    )

print("=" * 90)

del model
del backbone
del optimizer
del train_dataloader
del val_dataloader

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131
Backbone trainable parameters: 0
Probe trainable parameters:    49985


Probe Valid 1/120: 100%|██████████| 499/499 [00:41<00:00, 12.11it/s]



------------------------------------------------------------------------------------------
Epoch 001 | Mean Val MAE=0.087553 | Patience=0/20
block1       | train MAE 0.059681 | val MAE 0.102476 | 1-MAE 0.897524 | best 1-MAE 0.897524
block2       | train MAE 0.048225 | val MAE 0.086903 | 1-MAE 0.913097 | best 1-MAE 0.913097
block3       | train MAE 0.038492 | val MAE 0.089224 | 1-MAE 0.910776 | best 1-MAE 0.910776
block4       | train MAE 0.035916 | val MAE 0.086597 | 1-MAE 0.913403 | best 1-MAE 0.913403
block5       | train MAE 0.032246 | val MAE 0.085530 | 1-MAE 0.914470 | best 1-MAE 0.914470
block6       | train MAE 0.033047 | val MAE 0.084517 | 1-MAE 0.915483 | best 1-MAE 0.915483
block7       | train MAE 0.031586 | val MAE 0.085877 | 1-MAE 0.914123 | best 1-MAE 0.914123
block8       | train MAE 0.031946 | val MAE 0.089104 | 1-MAE 0.910896 | best 1-MAE 0.910896
block9       | train MAE 0.033814 | val MAE 0.085250 | 1-MAE 0.914750 | best 1-MAE 0.914750
block10      | train MAE 0.032

Probe Valid 2/120: 100%|██████████| 499/499 [00:42<00:00, 11.81it/s]



------------------------------------------------------------------------------------------
Epoch 002 | Mean Val MAE=0.086406 | Patience=0/20
block1       | train MAE 0.054425 | val MAE 0.091372 | 1-MAE 0.908628 | best 1-MAE 0.908628
block2       | train MAE 0.042062 | val MAE 0.086616 | 1-MAE 0.913384 | best 1-MAE 0.913384
block3       | train MAE 0.034414 | val MAE 0.089943 | 1-MAE 0.910057 | best 1-MAE 0.910776
block4       | train MAE 0.029945 | val MAE 0.086606 | 1-MAE 0.913394 | best 1-MAE 0.913403
block5       | train MAE 0.029645 | val MAE 0.086747 | 1-MAE 0.913253 | best 1-MAE 0.914470
block6       | train MAE 0.028666 | val MAE 0.085192 | 1-MAE 0.914808 | best 1-MAE 0.915483
block7       | train MAE 0.028856 | val MAE 0.084849 | 1-MAE 0.915151 | best 1-MAE 0.915151
block8       | train MAE 0.028126 | val MAE 0.085619 | 1-MAE 0.914381 | best 1-MAE 0.914381
block9       | train MAE 0.027090 | val MAE 0.086283 | 1-MAE 0.913717 | best 1-MAE 0.914750
block10      | train MAE 0.027

...


------------------------------------------------------------------------------------------
Epoch 036 | Mean Val MAE=0.086438 | Patience=18/20
block1       | train MAE 0.052804 | val MAE 0.090383 | 1-MAE 0.909617 | best 1-MAE 0.910338
block2       | train MAE 0.041662 | val MAE 0.090151 | 1-MAE 0.909849 | best 1-MAE 0.914118
block3       | train MAE 0.032772 | val MAE 0.086512 | 1-MAE 0.913488 | best 1-MAE 0.914749
block4       | train MAE 0.030051 | val MAE 0.085618 | 1-MAE 0.914382 | best 1-MAE 0.915615
block5       | train MAE 0.029250 | val MAE 0.084487 | 1-MAE 0.915513 | best 1-MAE 0.915528
block6       | train MAE 0.028360 | val MAE 0.087299 | 1-MAE 0.912701 | best 1-MAE 0.915483
block7       | train MAE 0.027346 | val MAE 0.086242 | 1-MAE 0.913758 | best 1-MAE 0.915429
block8       | train MAE 0.027240 | val MAE 0.085005 | 1-MAE 0.914995 | best 1-MAE 0.915506
block9       | train MAE 0.026443 | val MAE 0.084917 | 1-MAE 0.915083 | best 1-MAE 0.915955
block10      | train MAE 0.02

Probe Valid 37/120: 100%|██████████| 499/499 [00:41<00:00, 12.02it/s]



------------------------------------------------------------------------------------------
Epoch 037 | Mean Val MAE=0.088066 | Patience=19/20
block1       | train MAE 0.054347 | val MAE 0.096381 | 1-MAE 0.903619 | best 1-MAE 0.910338
block2       | train MAE 0.042282 | val MAE 0.085789 | 1-MAE 0.914211 | best 1-MAE 0.914211
block3       | train MAE 0.033783 | val MAE 0.086398 | 1-MAE 0.913602 | best 1-MAE 0.914749
block4       | train MAE 0.030810 | val MAE 0.088637 | 1-MAE 0.911363 | best 1-MAE 0.915615
block5       | train MAE 0.030343 | val MAE 0.088576 | 1-MAE 0.911424 | best 1-MAE 0.915528
block6       | train MAE 0.028767 | val MAE 0.088177 | 1-MAE 0.911823 | best 1-MAE 0.915483
block7       | train MAE 0.028543 | val MAE 0.087206 | 1-MAE 0.912794 | best 1-MAE 0.915429
block8       | train MAE 0.028638 | val MAE 0.087989 | 1-MAE 0.912011 | best 1-MAE 0.915506
block9       | train MAE 0.027549 | val MAE 0.086330 | 1-MAE 0.913670 | best 1-MAE 0.915955
block10      | train MAE 0.02

Probe Valid 38/120: 100%|██████████| 499/499 [00:41<00:00, 11.99it/s]


------------------------------------------------------------------------------------------
Epoch 038 | Mean Val MAE=0.088152 | Patience=20/20
block1       | train MAE 0.052868 | val MAE 0.101324 | 1-MAE 0.898676 | best 1-MAE 0.910338
block2       | train MAE 0.042958 | val MAE 0.091295 | 1-MAE 0.908705 | best 1-MAE 0.914211
block3       | train MAE 0.033666 | val MAE 0.087137 | 1-MAE 0.912863 | best 1-MAE 0.914749
block4       | train MAE 0.030646 | val MAE 0.085562 | 1-MAE 0.914438 | best 1-MAE 0.915615
block5       | train MAE 0.029363 | val MAE 0.090287 | 1-MAE 0.909713 | best 1-MAE 0.915528
block6       | train MAE 0.028784 | val MAE 0.086385 | 1-MAE 0.913615 | best 1-MAE 0.915483
block7       | train MAE 0.028486 | val MAE 0.087723 | 1-MAE 0.912277 | best 1-MAE 0.915429
block8       | train MAE 0.028391 | val MAE 0.084971 | 1-MAE 0.915029 | best 1-MAE 0.915506
block9       | train MAE 0.027466 | val MAE 0.086371 | 1-MAE 0.913629 | best 1-MAE 0.915955
block10      | train MAE 0.02